<a href="https://colab.research.google.com/github/jegazhu/SQL-AI-Agnt/blob/main/multi_turn_chat_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Su propósito, funcionalidad, uso y el impacto de integrar la IA en el análisis, consumo, validación y mitigación de problemas de datos:

Este cuaderno representa un paso fundamental en la integración de capacidades avanzadas de Inteligencia Artificial, específicamente Modelos de Lenguaje Grandes (LLMs), en nuestras operaciones de datos. El trabajo principal consistió en establecer una interfaz de IA conversacional robusta y resiliente, que permite la interacción directa con modelos potentes como Llama 3.1 8B Instruct.

El **propósito principal** de esta configuración es dotar a los profesionales de datos y a las partes interesadas de una herramienta intuitiva para la creación rápida de prototipos, la generación de consultas y la comprensión conceptual. En lugar de elaborar manualmente consultas SQL complejas o lidiar con construcciones de programación desconocidas, los usuarios pueden simplemente expresar sus necesidades en lenguaje natural.

Su **funcionalidad** se centra en proporcionar un bucle de chat interactivo que aprovecha el punto final de la API de NVIDIA. Hemos implementado características para la gestión segura de claves API, la transmisión en tiempo real de respuestas de IA y un manejo crucial de errores, incluyendo un 'Escudo HTML' para detectar e informar sobre límites de tasa de API o interrupciones del servicio. Esto asegura una experiencia de usuario fluida y confiable, incluso al interactuar con servicios externos.

En cuanto a su **uso**, este cuaderno sirve como un asistente dinámico. Los analistas de datos pueden solicitar consultas SQL para tareas específicas de extracción o manipulación de datos, pedir explicaciones de conceptos de datos complejos o incluso generar fragmentos de código Python para la limpieza y transformación de datos. El historial conversacional asegura que la IA mantenga el contexto, haciendo que las interacciones de múltiples turnos sean fluidas y productivas.

El **impacto** de implementar la IA en nuestros esfuerzos de análisis, consumo, validación y mitigación de problemas de datos será transformador:

*   **Análisis de Datos Acelerado**: Al automatizar la generación de consultas y código, el tiempo dedicado a las tareas rutinarias de preparación y exploración de datos se reducirá significativamente, permitiendo a los analistas centrarse en conocimientos de mayor valor.
*   **Acceso a Datos Democratizado**: Los usuarios no técnicos pueden interactuar con los datos de manera más efectiva, cerrando la brecha entre las preguntas de negocio y las operaciones técnicas de datos.
*   **Consumo de Datos Mejorado**: La IA puede proporcionar explicaciones y resúmenes de datos bajo demanda, haciendo que los conjuntos de datos complejos sean más comprensibles y accesibles para una audiencia más amplia.
*   **Validación de Datos Mejorada**: La IA puede ayudar a generar reglas de validación o identificar posibles anomalías a través del reconocimiento inteligente de patrones, reduciendo el esfuerzo manual y el error humano.
*   **Mitigación Proactiva de Problemas**: Al generar rápidamente consultas de diagnóstico o proponer soluciones basadas en los problemas de datos descritos, la IA puede ayudar a una identificación y mitigación más rápidas de los problemas de calidad de datos o de las tuberías.

En última instancia, esta iniciativa nos posiciona para aprovechar la IA como un multiplicador de fuerza, haciendo que nuestros flujos de trabajo de datos sean más eficientes, inteligentes y accesibles, fomentando así una cultura impulsada por los datos con una agilidad y una visión sin precedentes.

In [ ]:
# 1. Install the official OpenAI client library
!pip install openai -q

import os
import sys
from openai import OpenAI
from google.colab import userdata

# 2. Securely pull your NVIDIA key from Colab's Secrets manager
NVIDIA_API_KEY = userdata.get('NV_KEY')

# 3. Initialize the client targeting NVIDIA's cloud endpoint
client = OpenAI(
    base_url="https://nvidia.com",
    api_key=NVIDIA_API_KEY
)

# 4. Set up the conversation history array with a system instruction
conversation_history = [
    {
        "role": "system",
        "content": "You are a helpful, logical AI assistant. Provide direct, informative answers."
    }
]

print("💬 Llama 3.3 70B Live Chat Loop Initialized.")
print("Type 'exit', 'quit', or 'bye' to end the conversation.\n" + "-"*50)

# 5. Start the interactive multi-turn loop
while True:
    try:
        # Capture user input directly inside the notebook environment
        user_input = input("\nYou: ")

        # Check for break commands
        if user_input.strip().lower() in ['exit', 'quit', 'bye']:
            print("\n👋 Chat session ended. Goodbye!")
            break

        if not user_input.strip():
            continue

        # Append the new user message to maintain memory history
        conversation_history.append({"role": "user", "content": user_input})

        print("\nLlama-3.3-70B: ", end="")

        # Request a streaming response from the API
        stream = client.chat.completions.create(
            model="meta/llama-3.3-70b-instruct",
            messages=conversation_history,
            temperature=0.4,
            max_tokens=2048,
            stream=True  # Enables token-by-token streaming
        )

        # Print tokens to the notebook screen in real-time as they arrive
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                token = chunk.choices[0].delta.content
                print(token, end="")
                sys.stdout.flush()  # Forces immediate screen update in Colab
                full_response += token

        print() # Print a final newline after the stream finishes

        # Save the model's response back to the history so it remembers it next turn
        conversation_history.append({"role": "assistant", "content": full_response})

    except Exception as e:
        print(f"\n❌ Error during execution: {e}")
        print("Please check your internet connection or verify your 'NV_KEY' configuration.")
        break

In [ ]:
import os
import sys
from openai import OpenAI
from google.colab import userdata

# 1. Initialize client using your exact secret name
try:
    NVIDIA_API_KEY = userdata.get('NV_KEY')
    client = OpenAI(
        base_url="https://nvidia.com",
        api_key=NVIDIA_API_KEY
    )
except Exception:
    print("❌ Error: Double check that 'NV_KEY' is toggled ON in your Secrets (🔑) menu.")

conversation_history = [
    {"role": "system", "content": "You are a concise, helpful programming assistant."}
]

print("✨ Clean Chat Loop Ready. Type 'exit' to quit.\n" + "-"*50)

while True:
    try:
        user_input = input("\nYou: ").strip()

        if user_input.lower() in ['exit', 'quit', 'bye']:
            print("\n👋 Chat ended safely.")
            break

        if not user_input:
            continue

        conversation_history.append({"role": "user", "content": user_input})
        print("\nLlama-3.3-70B: ", end="")

        # Call API with explicit chunk processing safety
        stream = client.chat.completions.create(
            model="meta/llama-3.3-70b-instruct",
            messages=conversation_history,
            temperature=0.3,
            max_tokens=1024,
            stream=True
        )

        full_response = ""
        for chunk in stream:
            # Verify the chunk actually has text before attempting to print
            if chunk.choices and len(chunk.choices) > 0:
                token = chunk.choices[0].delta.content
                if token:  # Prevents printing None or empty lines infinitely
                    print(token, end="")
                    sys.stdout.flush()
                    full_response += token

        print() # End response line cleanly
        conversation_history.append({"role": "assistant", "content": full_response})

    except KeyboardInterrupt:
        print("\n\n🛑 Loop stopped manually by user.")
        break
    except Exception as e:
        print(f"\n❌ Error during generation: {e}")
        break

In [ ]:
import os
import sys
from openai import OpenAI
from google.colab import userdata

# 1. Initialize client using your exact secret name
try:
    NVIDIA_API_KEY = userdata.get('NV_KEY')
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",  # FIX: was "https://nvidia.com"
        api_key=NVIDIA_API_KEY,
        timeout=60.0,      # fail after 60s instead of silently hanging up to 600s
        max_retries=1,     # don't stack up long retry backoffs
    )
except Exception:
    print("❌ Error: Double check that 'NV_KEY' is toggled ON in your Secrets (🔑) menu.")

conversation_history = [
    {"role": "system", "content": "You are a concise, helpful programming assistant."}
]

print("🛡️ HTML-Shielded Chat Loop Ready. Type 'exit' to quit.\n" + "-"*50)

while True:
    try:
        user_input = input("\nYou: ").strip()

        if user_input.lower() in ['exit', 'quit', 'bye']:
            print("\n👋 Chat ended safely.")
            break

        if not user_input:
            continue
        conversation_history.append({"role": "user", "content": user_input})
        print("\nLlama-3.3-70B: ", end="")
        sys.stdout.flush()

        # 2. Call API with explicit stream processing
        import time
        t0 = time.time()
        first_token_seen = False
        stream = client.chat.completions.create(
            model="meta/llama-3.3-70b-instruct",
            messages=conversation_history,
            temperature=0.3,
            max_tokens=1024,
            stream=True
        )

        full_response = ""
        for chunk in stream:
            if not first_token_seen:
                first_token_seen = True
                # (debug) uncomment to see time-to-first-token:
                # print(f"[{time.time()-t0:.1f}s to first chunk] ", end="")
            if chunk.choices and len(chunk.choices) > 0:
                token = chunk.choices[0].delta.content  # FIX: was chunk.choices.delta.content
                if token:
                    # 🛡️ HTML SHIELD: Detect if server returned a web error page instead of text tokens
                    if token.strip().startswith("<!DOCTYPE") or token.strip().startswith("<html") or token.strip().startswith("<div"):
                        print("\n\n⚠️ NVIDIA Server returned an HTML web error page instead of text.")
                        print("👉 Reason: You likely hit a rate limit or ran out of free credits.")
                        full_response = "Error: Server returned HTML."
                        break  # Break out of stream immediately to save your buffer from crashing

                    print(token, end="")
                    sys.stdout.flush()
                    full_response += token

        print()  # End response line cleanly

        # Only preserve successful turns in history
        if "Error: Server returned HTML" not in full_response:
            conversation_history.append({"role": "assistant", "content": full_response})
    except KeyboardInterrupt:
        print("\n\n🛑 Loop stopped manually by user.")
        break
    except Exception as e:
        print(f"\n❌ Error during generation: {e}")
        break

In [10]:
import time
from openai import OpenAI
from google.colab import userdata

NVIDIA_API_KEY = userdata.get('NV_KEY')
print("Key loaded:", bool(NVIDIA_API_KEY), "| length:", len(NVIDIA_API_KEY) if NVIDIA_API_KEY else 0)

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
    timeout=20.0,     # fail fast instead of hanging up to 600s
    max_retries=0,    # no silent retry delay
)

print("Sending test request...")
start = time.time()
try:
    resp = client.chat.completions.create(
        model="meta/llama-3.3-70b-instruct",
        messages=[{"role": "user", "content": "Say OK"}],
        max_tokens=10,
        stream=False,   # non-streaming first, simpler to debug
    )
    print(f"✅ Success in {time.time()-start:.1f}s")
    print(resp.choices[0].message.content)
except Exception as e:
    print(f"❌ Failed after {time.time()-start:.1f}s")
    print(type(e).__name__, "-", e)

Key loaded: True | length: 70
Sending test request...
✅ Success in 11.0s
OK




> **mistralai/mistral-large-3-675b-instruct-2512**

Este código Python configura un bucle de chat interactivo en Google Colab, lo que permite conversar con un modelo de lenguaje grande alojado en el punto final de la API de NVIDIA. Analicemos cada parte:

**Declaraciones de importación:** Importa los módulos necesarios, como `sys` para parámetros y funciones específicas del sistema, `time` para medir la duración de la ejecución, `OpenAI` de la biblioteca `openai` para interactuar con la API de LLM y `userdata` de `google.colab` para acceder de forma segura a las claves de la API.

**Inicialización del cliente de la API:** Esta sección recupera la clave de la API de NVIDIA (NV_KEY) del Administrador de secretos de Colab. A continuación, inicializa un cliente de OpenAI, apuntando su `base_url` al punto final de la API de NVIDIA (https://integrate.api.nvidia.com/v1). También establece un tiempo de espera de 60 segundos y un máximo de reintentos (`max_retries`) de 1 para evitar bloqueos indefinidos y controlar el comportamiento de los reintentos. Se incluye un bloque `try-except` para detectar posibles errores si la clave de la API no está configurada correctamente.

Selección de modelo: La variable MODEL_ID especifica el modelo de lenguaje que utilizará el chat; en este caso, meta/llama-3.1-8b-instruct. También se incluye una sugerencia de modelo alternativo comentada.

**Historial de conversación:** conversation_history es una lista que almacena toda la sesión de chat. Comienza con un mensaje del sistema que define el rol de la IA (un asistente de programación conciso y útil). Este historial es crucial para que el modelo mantenga el contexto a lo largo de varias conversaciones.

**Bucle de chat interactivo (mientras sea verdadero):** Este es el núcleo de la aplicación, que se ejecuta continuamente hasta que se detiene explícitamente:

**Solicitud de entrada del usuario:** Solicita al usuario que introduzca información (Tú: ) y captura su respuesta.

**Condiciones de salida:** Comprueba si el usuario escribe «salir», «quit» o «bye» para finalizar el bucle de chat de forma segura.

**Añadir mensaje del usuario:** La entrada del usuario se añade a la lista conversation_history con el rol «usuario».

**Llamada a la API para respuesta en tiempo real:** A continuación, se llama al método `client.chat.completions.create`.

**Aquí es donde ocurre la magia:** se envía el historial de conversación al ID de modelo especificado con un nivel de creatividad de 0,3, un límite de tokens máximos y `stream=True`. El modo en tiempo real implica que el modelo devuelve los tokens (partes de su respuesta) uno a uno, lo que permite su visualización en tiempo real.

**Gestión de errores para el ID de modelo:** Un bloque `try-except` alrededor de la llamada a la API captura específicamente los errores si el ID de modelo no es válido o no está disponible, informando al usuario y eliminando el turno fallido del historial.

**Procesamiento de flujo y protección HTML:** Se itera a través del flujo de tokens recibidos de la API:

Cada token se imprime en la consola a medida que llega, y `sys.stdout.flush()` garantiza su visualización inmediata.

Una función de protección HTML comprueba si el token entrante comienza con etiquetas HTML. Esta es una forma ingeniosa de detectar si el servidor NVIDIA devolvió una página de error (por ejemplo, debido a la limitación de velocidad o a créditos caducados) en lugar de una respuesta de texto válida. Si se detecta HTML, se imprime una advertencia y se interrumpe la transmisión.

La función `full_response` acumula todos los tokens para almacenar la respuesta completa de la IA.

**Añadir respuesta de la IA:** Una vez que la IA ha terminado de responder, su respuesta completa se añade al historial de conversaciones con el rol de "asistente", manteniendo el contexto de la conversación para los turnos posteriores.

**Gestión global de excepciones:** Existen bloques `try-except` para gestionar las interrupciones por teclado (cuando el usuario detiene manualmente la ejecución, por ejemplo, pulsando Ctrl+C) y cualquier otra excepción general que pueda ocurrir durante el ciclo de chat, proporcionando mensajes de error informativos.

In [ ]:
# @title Multi-turn Chat
import sys
import time
from openai import OpenAI
from google.colab import userdata

# --- Reuse the same client setup pattern that already works ---
try:
    NVIDIA_API_KEY = userdata.get('NV_KEY')
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=NVIDIA_API_KEY,
        timeout=60.0,
        max_retries=1,
    )
except Exception:
    print("❌ Error: Double check that 'NV_KEY' is toggled ON in your Secrets (🔑) menu.")

# --- Model selection ---
# Primary target: Mistral Large 2 (Mistral-Large-Instruct-2407)
MODEL_ID = "meta/llama-3.1-8b-instruct"
# Fallback if the above 404s on your account (newer flagship on NVIDIA's catalog):
# MODEL_ID = "mistralai/mistral-large-3-675b-instruct-2512"

conversation_history = [
    {"role": "system", "content": "You are a concise, helpful programming assistant."}
]

print(f"🛡️ HTML-Shielded Chat Loop Ready ({MODEL_ID}). Type 'exit' to quit.\n" + "-"*50)

while True:
    try:
        user_input = input("\nYou: ").strip()

        if user_input.lower() in ['exit', 'quit', 'bye']:
            print("\n👋 Chat ended safely.")
            break

        if not user_input:
            continue
        conversation_history.append({"role": "user", "content": user_input})
        print(f"\n{MODEL_ID}: ", end="")
        sys.stdout.flush()

        t0 = time.time()
        try:
            stream = client.chat.completions.create(
                model=MODEL_ID,
                messages=conversation_history,
                temperature=0.3,
                max_tokens=1024,
                stream=True
            )
        except Exception as e:
            # Catch a bad/unsupported model id explicitly instead of hanging or crashing the loop
            print(f"\n❌ Could not reach model '{MODEL_ID}': {e}")
            print("👉 If this is a 404, the model id may have changed — check its card at build.nvidia.com.")
            conversation_history.pop()  # remove the user turn that failed
            continue

        full_response = ""
        for chunk in stream:
            if chunk.choices and len(chunk.choices) > 0:
                token = chunk.choices[0].delta.content
                if token:
                    if token.strip().startswith("<!DOCTYPE") or token.strip().startswith("<html") or token.strip().startswith("<div"):
                        print("\n\n⚠️ NVIDIA Server returned an HTML web error page instead of text.")
                        print("👉 Reason: You likely hit a rate limit, ran out of free credits, or the base_url is wrong.")
                        full_response = "Error: Server returned HTML."
                        break

                    print(token, end="")
                    sys.stdout.flush()
                    full_response += token

        print()

        if "Error: Server returned HTML" not in full_response:
            conversation_history.append({"role": "assistant", "content": full_response})
    except KeyboardInterrupt:
        print("\n\n🛑 Loop stopped manually by user.")
        break
    except Exception as e:
        print(f"\n❌ Error during generation: {e}")
        break

🛡️ HTML-Shielded Chat Loop Ready (meta/llama-3.1-8b-instruct). Type 'exit' to quit.
--------------------------------------------------

You: Provide a production-ready system query for PostgreSQL to identify the top 5 longest-running active queries right now, including their execution state, pid, and how long they have been blocking other system resources.

meta/llama-3.1-8b-instruct: You can use the following PostgreSQL query to identify the top 5 longest-running active queries:

```sql
SELECT 
    pg_stat_activity.query,
    pg_stat_activity.state AS execution_state,
    pg_stat_activity.pid,
    pg_stat_activity.query_start,
    AGE(pg_stat_activity.query_start) AS query_duration,
    pg_locks.pid AS blocking_pid,
    pg_locks.granted AS is_blocked
FROM 
    pg_stat_activity
JOIN 
    pg_locks ON pg_locks.pid = pg_stat_activity.pid
WHERE 
    pg_stat_activity.state IN ('active', 'idle in transaction')
    AND pg_locks.pid IS NOT NULL
ORDER BY 
    AGE(pg_stat_activity.query_start) D